# OCR models on Google Colab

Run this notebook from VS Code using your Google Colab extension. Use a GPU runtime, mount Google Drive, and keep input/output paths in Drive.

Expected input folder: `/content/drive/MyDrive/tubitak_sample_docs`

Output base folder: `/content/drive/MyDrive/output/output_ocr`

Each model writes into its own subfolder. Each subfolder is cleared at the start of the run and ends with one ZIP file containing that model's JSON/Markdown/images/assets.

In [ ]:
# Check that the Colab runtime has a GPU.
!nvidia-smi || true

In [ ]:
# Install runtime dependencies. Run this once per fresh Colab runtime.
%pip install -q -U paddleocr
%pip install -q "transformers==4.46.3" "tokenizers==0.20.3" einops addict easydict safetensors accelerate

# DeepSeek recommends flash-attn for GPU inference. This can fail on some Colab images,
# so the DeepSeek cell below falls back if flash attention is unavailable.
!pip install -q flash-attn==2.7.3 --no-build-isolation || true

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

INPUT_DIR = Path('/content/drive/MyDrive/tubitak_sample_docs')
OUTPUT_BASE_DIR = Path('/content/drive/MyDrive/output/output_ocr')
PADDLE_VL_OUTPUT_DIR = OUTPUT_BASE_DIR / 'paddleocr_vl_1_6'
DEEPSEEK_OUTPUT_DIR = OUTPUT_BASE_DIR / 'deepseek_ocr_2'

for output_dir in [PADDLE_VL_OUTPUT_DIR, DEEPSEEK_OUTPUT_DIR]:
    output_dir.mkdir(parents=True, exist_ok=True)
    for existing_file in output_dir.iterdir():
        if existing_file.is_file():
            existing_file.unlink()

print(f'Input directory: {INPUT_DIR}')
print(f'Output base directory: {OUTPUT_BASE_DIR}')
print(f'PaddleOCR-VL output directory: {PADDLE_VL_OUTPUT_DIR}')
print(f'DeepSeek output directory: {DEEPSEEK_OUTPUT_DIR}')
print(f'Input exists: {INPUT_DIR.exists()}')

## Run PaddleOCR-VL-1.6

This cell reuses the existing PaddleOCR-VL pipeline when rerun in the same Colab runtime. If you need to change `batch_size` or `pipeline_version`, restart the runtime first.

In [ ]:
import os
import shutil
import tempfile
import time
from datetime import datetime
from paddleocr import PaddleOCRVL

start_time = time.perf_counter()
print(f'[DEBUG] Started PaddleOCR-VL at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Output directory: {PADDLE_VL_OUTPUT_DIR}')

if 'paddle_vl_pipeline' not in globals():
    paddle_vl_pipeline = PaddleOCRVL(batch_size=1, pipeline_version='v1.6')
    print('[DEBUG] Initialized PaddleOCR-VL pipeline')
else:
    print('[DEBUG] Reusing existing PaddleOCR-VL pipeline to avoid PDX reinitialization')

combined_output_file = PADDLE_VL_OUTPUT_DIR / 'paddleocr_vl_1_6_output.zip'

with tempfile.TemporaryDirectory() as staging_dir:
    results = paddle_vl_pipeline.predict(str(INPUT_DIR))

    for index, res in enumerate(results, start=1):
        result_start_time = time.perf_counter()
        res.print()
        res.save_to_json(save_path=staging_dir)
        res.save_to_markdown(save_path=staging_dir)
        print(f'[DEBUG] PaddleOCR-VL result {index} saved to staging in {time.perf_counter() - result_start_time:.2f}s')

    shutil.make_archive(str(combined_output_file.with_suffix('')), 'zip', staging_dir)

print(f'[DEBUG] Saved single PaddleOCR-VL ZIP output to: {combined_output_file}')

print(f'[DEBUG] Finished PaddleOCR-VL at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Total elapsed time: {time.perf_counter() - start_time:.2f}s')

## Run DeepSeek-OCR-2

DeepSeek-OCR-2 is much heavier. If this cell runs out of memory, use a larger Colab GPU runtime. If folder input fails in your runtime, set `IMAGE_FILE` to one image/PDF path inside `INPUT_DIR`.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import shutil
import tempfile
import time
from datetime import datetime

import torch
from transformers import AutoModel, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError('DeepSeek-OCR-2 needs a GPU runtime. In Colab, select Runtime > Change runtime type > GPU.')
model_name = 'deepseek-ai/DeepSeek-OCR-2'
prompt = '<image>\n<|grounding|>Convert the document to markdown. '

# Use the whole folder by default, matching your local script.
# If needed, change this to a single file, e.g. str(INPUT_DIR / 'sample.jpg').
IMAGE_FILE = str(INPUT_DIR)

start_time = time.perf_counter()
print(f'[DEBUG] Started DeepSeek OCR at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Output directory: {DEEPSEEK_OUTPUT_DIR}')

tokenizer_start_time = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
print(f'[DEBUG] Tokenizer loaded in {time.perf_counter() - tokenizer_start_time:.2f}s')

model_start_time = time.perf_counter()
try:
    model = AutoModel.from_pretrained(
        model_name,
        _attn_implementation='flash_attention_2',
        trust_remote_code=True,
        use_safetensors=True,
        low_cpu_mem_usage=True,
    )
except Exception as exc:
    print(f'[DEBUG] Flash attention load failed; falling back. Reason: {exc}')
    model = AutoModel.from_pretrained(
        model_name,
        trust_remote_code=True,
        use_safetensors=True,
        low_cpu_mem_usage=True,
    )

model = model.eval().cuda().to(torch.bfloat16)
print(f'[DEBUG] Model loaded in {time.perf_counter() - model_start_time:.2f}s')

infer_start_time = time.perf_counter()
combined_output_file = DEEPSEEK_OUTPUT_DIR / 'deepseek_ocr_2_output.zip'

with tempfile.TemporaryDirectory() as staging_dir:
    res = model.infer(
        tokenizer,
        prompt=prompt,
        image_file=IMAGE_FILE,
        output_path=staging_dir,
        base_size=1024,
        image_size=768,
        crop_mode=True,
        save_results=True,
    )

    if res is not None:
        with open(os.path.join(staging_dir, 'deepseek_return_value.md'), 'w', encoding='utf-8') as f:
            f.write('# DeepSeek-OCR-2 Return Value\n\n')
            f.write(str(res))

    if not os.listdir(staging_dir):
        with open(os.path.join(staging_dir, 'no_output.md'), 'w', encoding='utf-8') as f:
            f.write('DeepSeek-OCR-2 did not write output files and returned no value.')

    shutil.make_archive(str(combined_output_file.with_suffix('')), 'zip', staging_dir)

print(f'[DEBUG] Saved single DeepSeek OCR ZIP output to: {combined_output_file}')

print(f'[DEBUG] Inference completed in {time.perf_counter() - infer_start_time:.2f}s')
print(f'[DEBUG] Finished DeepSeek OCR at {datetime.now().isoformat(timespec="seconds")}')
print(f'[DEBUG] Total elapsed time: {time.perf_counter() - start_time:.2f}s')